In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.datasets import fetch_openml

print("Fetching dataset...")
german = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
X = german.data
y = german.target
y = y.map({'good': 1, 'bad': 0})

categorical_cols = X.select_dtypes(include=['category', 'object']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

X_preprocessed = preprocessor.fit_transform(X)
feature_names = (numerical_cols.tolist() + preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist())
X_df = pd.DataFrame(X_preprocessed, columns=feature_names)
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.30, random_state=84)

print("--- PART 1: MODEL BUILD & COMPARISON ---")
model_a = LinearRegression().fit(X_train, y_train)
preds_a = model_a.predict(X_test)
class_a = (preds_a >= np.median(preds_a)).astype(int)
model_b = LogisticRegression(max_iter=1000).fit(X_train, y_train)
probs_b = model_b.predict_proba(X_test)[:, 1]
class_b = (probs_b >= 0.5).astype(int)
print(f"Total Mismatched Cases: {np.sum(class_a != class_b)} ({(np.sum(class_a != class_b) / len(y_test)) * 100:.2f}%)\n")

print("--- PART 2.1: SIMPSON'S PARADOX ---")
age_idx = numerical_cols.get_loc('age')
age_mean = preprocessor.named_transformers_['num'].mean_[age_idx]
age_scale = preprocessor.named_transformers_['num'].scale_[age_idx]
scaled_age_30 = (30 - age_mean) / age_scale
subgroup_mask = X_train['age'] > scaled_age_30

log_full = LogisticRegression(max_iter=1000).fit(X_train, y_train)
log_A = LogisticRegression(max_iter=1000).fit(X_train[subgroup_mask], y_train[subgroup_mask])
log_B = LogisticRegression(max_iter=1000).fit(X_train[~subgroup_mask], y_train[~subgroup_mask])
idx = feature_names.index('credit_amount')
print(f"Credit Amount Coefs -> Full: {log_full.coef_[0][idx]:.4f}, Age>30: {log_A.coef_[0][idx]:.4f}, Age<=30: {log_B.coef_[0][idx]:.4f}\n")

print("--- PART 2.2: OMITTED-VARIABLE BIAS ---")
X_train_reduced = X_train.drop('duration', axis=1)
log_reduced = LogisticRegression(max_iter=1000).fit(X_train_reduced, y_train)
print("Coefficient Shifts (Full vs Reduced):")
features_to_check = ['credit_amount', 'age', 'installment_commitment', 'residence_since']
for feat in features_to_check:
    idx_full = feature_names.index(feat)
    idx_red = X_train_reduced.columns.get_loc(feat)
    c_full = log_full.coef_[0][idx_full]
    c_red = log_reduced.coef_[0][idx_red]
    pct_change = ((c_red - c_full) / abs(c_full)) * 100
    print(f"  {feat}: Full = {c_full:.4f}, Reduced = {c_red:.4f} | Shift = {pct_change:.2f}%")
print()

print("--- PART 2.3: ADVERSARIAL SUBSET ---")
adv_mask = ((probs_b >= 0.75) & (y_test == 0)) | ((probs_b <= 0.25) & (y_test == 1))
X_adv = X_test[adv_mask]
print(f"Adversarial examples found: {len(X_adv)} ({(len(X_adv)/len(y_test))*100:.2f}% of test set)")
if len(X_adv) > 0:
    print(f"Average Duration (scaled) in Subset: {X_adv['duration'].mean():.4f} vs Test Set: {X_test['duration'].mean():.4f}")
    print(f"Average Credit Amount (scaled) in Subset: {X_adv['credit_amount'].mean():.4f} vs Test Set: {X_test['credit_amount'].mean():.4f}\n")

print("--- PART 2.4: FAIRNESS TRADE-OFF ---")
test_mask = X_test['age'] > scaled_age_30
y_A = y_test[test_mask]
y_B = y_test[~test_mask]
probs_A = model_b.predict_proba(X_test[test_mask])[:, 1]
probs_B = model_b.predict_proba(X_test[~test_mask])[:, 1]

thresholds = np.arange(0.05, 1.05, 0.1)
print(f"{'Threshold':<10} | {'DP':<10} | {'EO':<10} | {'PP'}")
print("-" * 45)
for t in thresholds:
    pred_A = (probs_A >= t).astype(int)
    pred_B = (probs_B >= t).astype(int)
    p_pred1_A = np.mean(pred_A)
    p_pred1_B = np.mean(pred_B)
    dp = p_pred1_A / p_pred1_B if p_pred1_B > 0 else 0
    tpr_A = np.sum((pred_A == 1) & (y_A == 1)) / np.sum(y_A == 1) if np.sum(y_A == 1) > 0 else 0
    tpr_B = np.sum((pred_B == 1) & (y_B == 1)) / np.sum(y_B == 1) if np.sum(y_B == 1) > 0 else 0
    fpr_A = np.sum((pred_A == 1) & (y_A == 0)) / np.sum(y_A == 0) if np.sum(y_A == 0) > 0 else 0
    fpr_B = np.sum((pred_B == 1) & (y_B == 0)) / np.sum(y_B == 0) if np.sum(y_B == 0) > 0 else 0
    eo_tpr = tpr_A / tpr_B if tpr_B > 0 else (1.0 if tpr_A == tpr_B else 0)
    eo_fpr = fpr_A / fpr_B if fpr_B > 0 else (1.0 if fpr_A == fpr_B else 0)
    eo = min(eo_tpr, eo_fpr)
    pp_A = np.sum((pred_A == 1) & (y_A == 1)) / np.sum(pred_A == 1) if np.sum(pred_A == 1) > 0 else 0
    pp_B = np.sum((pred_B == 1) & (y_B == 1)) / np.sum(pred_B == 1) if np.sum(pred_B == 1) > 0 else 0
    pp = pp_A / pp_B if pp_B > 0 else 0
    print(f"{t:<10.2f} | {dp:<10.4f} | {eo:<10.4f} | {pp:.4f}")

Fetching dataset...
--- PART 1: MODEL BUILD & COMPARISON ---
Total Mismatched Cases: 54 (18.00%)

--- PART 2.1: SIMPSON'S PARADOX ---
Credit Amount Coefs -> Full: -0.3664, Age>30: -0.3703, Age<=30: -0.3295

--- PART 2.2: OMITTED-VARIABLE BIAS ---
Coefficient Shifts (Full vs Reduced):
  credit_amount: Full = -0.3664, Reduced = -0.5245 | Shift = -43.15%
  age: Full = 0.1084, Reduced = 0.1146 | Shift = 5.74%
  installment_commitment: Full = -0.3802, Reduced = -0.4260 | Shift = -12.05%
  residence_since: Full = 0.0880, Reduced = 0.0935 | Shift = 6.31%

--- PART 2.3: ADVERSARIAL SUBSET ---
Adversarial examples found: 20 (6.67% of test set)
Average Duration (scaled) in Subset: 0.5307 vs Test Set: 0.0570
Average Credit Amount (scaled) in Subset: 0.5554 vs Test Set: 0.0363

--- PART 2.4: FAIRNESS TRADE-OFF ---
Threshold  | DP         | EO         | PP
---------------------------------------------
0.05       | 1.0000     | 1.0000     | 1.0852
0.15       | 0.9939     | 0.9535     | 1.0901
0.25  